# Simple mixed-kernel breast-cancer example

This notebook uses the local Wisconsin Diagnostic Breast Cancer data to fit a weighted sum of linear and RBF kernels. The mixture weights, RBF gamma, and SVC cost are tuned together. Forward selection treats each `(kernel, feature)` assignment independently, so a measurement can be active in one kernel, both kernels, or neither.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from mistic import MixedKernel, combined_rank, cvSet, kernelWrapper, paramSet, score_svc, svmSet

## Load the example data

The model uses all 30 tumor measurements: ten means, ten standard errors, and ten worst-value measurements. Malignant tumors are the positive class.

In [ ]:
properties = [
    "Radius", "Texture", "Perimeter", "Area", "Smoothness",
    "Compactness", "Concavity", "Concave Points", "Symmetry",
    "Fractal Dimension",
]
column_names = (
    ["ID", "Diagnosis"]
    + [f"Mean {name}" for name in properties]
    + [f"{name} SE" for name in properties]
    + [f"Worst {name}" for name in properties]
)
data = pd.read_csv("wdbc.data", header=None, names=column_names)
feature_names = column_names[2:]
X = data.loc[:, feature_names].to_numpy(dtype=float)
y = (data["Diagnosis"] == "M").to_numpy(dtype=int)
print(f"Samples: {len(y)}, features: {X.shape[1]}, malignant: {y.sum()}")

## Development and blind sets

The scaler is fitted only on the development partition. The blind partition is not used during cross-validation, tuning, or feature selection.

In [ ]:
X_dev, X_blind, y_dev, y_blind = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=7
)
scaler = StandardScaler().fit(X_dev)
X_dev = scaler.transform(X_dev)
X_blind = scaler.transform(X_blind)

splits = cvSet(X_dev, y_dev)
splits.classification(num_sets=3, validation_size=0.20, random_seed=7)

## Define the mixed kernel and tuning grid

Named leaf parameters use the `leaf_name__parameter` form, such as `radial__gamma`. Named mixture weights are ordinary tunable kernel hyperparameters. Both kernels may initially consider every feature.

In [ ]:
kernel = MixedKernel.weighted_sum(
    [
        kernelWrapper("linear", name="linear"),
        kernelWrapper("rbf", name="radial"),
    ],
    weight_parameters=["linear_weight", "rbf_weight"],
)

gamma_scale = 1.0 / (X_dev.shape[1] * X_dev.var())
parameter_grid = [
    paramSet(
        {"C": C},
        {
            "radial__gamma": gamma_scale * gamma_factor,
            "linear_weight": linear_weight,
            "rbf_weight": 1.0 - linear_weight,
        },
    )
    for C in (0.5, 2.0)
    for gamma_factor in (0.5, 2.0)
    for linear_weight in (0.25, 0.50, 0.75)
]
print(f"gamma='scale': {gamma_scale:.4f}; candidates: {len(parameter_grid)}")

## Kernel-aware forward selection

Singleton perturbation sets make each original measurement independently selectable. With `kernel_feature_selection="independent"`, MISTIC creates a separate candidate for every kernel-feature pair. Sensitivity ranking is averaged across the cross-validation models.

In [ ]:
candidate_features = np.arange(X_dev.shape[1])
model = svmSet(
    SVC(kernel="precomputed", class_weight="balanced", tol=1e-9),
    splits,
    score_svc(weight=0.5).score,
    kernel=kernel,
    kernel_feature_selection="independent",
    perturbation_sets=[[int(i)] for i in candidate_features],
    kernel_candidate_features={
        "linear": candidate_features,
        "radial": candidate_features,
    },
)
model.tune_models(parameter_grid)
print("Full-feature mixed-kernel parameters:", model.parameters_.model, model.parameters_.kernel)
print(f"Full-feature blind ROC AUC: {roc_auc_score(y_blind, model.decision_function(X_blind)):.3f}")

model.greedy_forward_selection(
    parameter_grid,
    max_features=15,
    num_initial_medoids=1,
    addition_factor=0.1,
    feature_ranker=combined_rank(weight=0.75).compute,
    set_for_rank="train",
    post_find_knee=False,
)

In [ ]:
selected_units = pd.DataFrame([
    {
        "kernel": model.selection_space_[index].kernel_id,
        "feature": feature_names[model.selection_space_[index].features[0]],
    }
    for index in model.selected_selection_units_
])
display(selected_units)
print("Selected model parameters:", model.parameters_.model)
print("Selected kernel parameters:", model.parameters_.kernel)
print("Kernel-specific feature indices:",
      {name: values.tolist() for name, values in model.kernel_features.items()})

In [ ]:
model.plot_performance(metric="score", count="num_selection_features")
plt.title("Mixed-kernel forward-selection performance")
plt.xlabel("Kernel-feature assignments")
plt.show()

## Blind performance and feature attribution

In [ ]:
blind_score = model.decision_function(X_blind)
blind_prediction = model.predict(X_blind)
print(f"Blind ROC AUC: {roc_auc_score(y_blind, blind_score):.3f}")
print(f"Blind balanced accuracy: {balanced_accuracy_score(y_blind, blind_prediction):.3f}")
print("Confusion matrix:\n", confusion_matrix(y_blind, blind_prediction))

## Integrated-gradients attributions

Attributions are computed for the complete blind set. The development-set mean is supplied explicitly as a common reference point, so each attribution describes the model-score change from an average standardized training sample to the blind sample. For the mixed kernel, MISTIC differentiates the complete selected kernel expression and returns attributions aligned with the original feature names.

In [ ]:
ig_result = model.explain_integrated_gradients(
    X_blind,
    feature_names=feature_names,
    target=y_blind,
    reference_point=np.mean(X_dev, axis=0),
    num_steps=30,
)

mean_absolute_attribution = ig_result.to_frame().abs().mean().sort_values(ascending=False)
mean_absolute_attribution.plot.bar(figsize=(8, 3), ylabel="Mean |integrated gradient|")
plt.title("Blind-set attribution magnitude")
plt.tight_layout()
plt.show()
mean_absolute_attribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ig_result.heatmap(
    ax=ax,
    cluster=True,
    cmap="coolwarm",
    imshow_kwargs={"rasterized": True},
    target_strip_width=0.20,
    strip_pad=0.25,
    colorbar_width=0.16,
    colorbar_pad=0.20,
    colorbar_gap=0.60,
)
ax.set_title("Clustered blind-set integrated gradients")
ax.tick_params(axis="x", labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ig_result.summary_plot(
    ax=ax,
    max_features=20,
    cmap="coolwarm",
    random_state=0,
    scatter_kwargs={"s": 28, "alpha": 0.70, "edgecolors": "none"},
)
ax.set_title("Integrated-gradient feature summary")
ax.set_xlabel("Integrated gradient")
plt.tight_layout()
plt.show()

### Attribution-based interaction view

With no feature pair supplied, the result selects two features using its attribution-based interaction heuristic. This is an exploratory attribution view rather than a formal statistical interaction test.

In [ ]:
ig_result.interaction_plot(
    scatter_kwargs={"s": 38, "alpha": 0.75, "edgecolors": "none"},
)
plt.show()